# Sola Face LoRA — Kohya single-process (Fooocus-compatible)

1. Runtime → **GPU**
2. Upload `datasets/sola_face_kohya.zip`
3. Run cells in order
4. Download `sola_face_sdxl.safetensors`

Trigger: `sola_face`


In [ ]:
# @title 0) GPU check
!nvidia-smi
import torch, sys
print(sys.version)
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → T4/L4"
print("OK", torch.cuda.get_device_name(0))


In [ ]:
# @title 1) Install Kohya sd-scripts (pinned)
import os, subprocess, sys
os.chdir("/content")
!pip -q install -U pip setuptools wheel
!pip -q install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121

if os.path.isdir("/content/sd-scripts"):
    import shutil; shutil.rmtree("/content/sd-scripts")
!git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git /content/sd-scripts
os.chdir("/content/sd-scripts")

# Minimal deps that usually work on Colab
!pip -q install accelerate==0.33.0 transformers==4.44.2 diffusers==0.30.3 safetensors ftfy einops opencv-python-headless
!pip -q install bitsandbytes==0.43.3 prodigyopt lion-pytorch voluptuous toml
!pip -q install xformers==0.0.28.post1 --index-url https://download.pytorch.org/whl/cu121 || pip -q install xformers

print("sd-scripts OK", os.listdir('.')[:10])
assert os.path.isfile("sdxl_train_network.py"), "sdxl_train_network.py missing"
assert os.path.isdir("networks"), "networks/ missing"


In [ ]:
# @title 2) Upload zip
import os, zipfile, shutil
from google.colab import files

DATA = "/content/sola_data"
shutil.rmtree(DATA, ignore_errors=True)
os.makedirs(DATA, exist_ok=True)
os.chdir(DATA)

print("Upload foocus_new/datasets/sola_face_kohya.zip")
uploaded = files.upload()
assert uploaded

for name in uploaded:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(os.path.join(DATA, name)) as z:
            z.extractall(DATA)
            print("entries", len(z.namelist()), z.namelist()[:3])

found = None
for root, dirs, files_ in os.walk(DATA):
    for d in dirs:
        if d == "10_sola_face":
            found = os.path.join(root, d)
            break
    if found:
        break

if not found:
    jpgs = [os.path.join(r, f) for r, _, fs in os.walk(DATA) for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    assert len(jpgs) >= 10, f"bad zip images={len(jpgs)}"
    found = os.path.join(DATA, "10_sola_face")
    os.makedirs(found, exist_ok=True)
    for src in jpgs:
        stem = os.path.splitext(os.path.basename(src))[0]
        shutil.copy2(src, os.path.join(found, stem + ".jpg"))
        tsrc = os.path.splitext(src)[0] + ".txt"
        tdst = os.path.join(found, stem + ".txt")
        if os.path.isfile(tsrc):
            shutil.copy2(tsrc, tdst)
        else:
            open(tdst, "w").write("sola_face, photo of a woman, looking at camera\n")

TRAIN_ROOT = os.path.dirname(found)
OUT = "/content/outputs/sola_face_lora"
os.makedirs(OUT, exist_ok=True)
n = len([f for f in os.listdir(found) if f.lower().endswith(".jpg")])
print("TRAIN_ROOT", TRAIN_ROOT)
print("CLASS", found, "jpg", n)
assert n >= 10


In [ ]:
# @title 3) Train (python direct, NO accelerate launch)
import os, sys, subprocess

os.chdir("/content/sd-scripts")
LOG = "/content/outputs/sola_face_lora/train.log"

# T4-safe defaults
cmd = [
    sys.executable, "sdxl_train_network.py",
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
    f"--train_data_dir={TRAIN_ROOT}",
    f"--output_dir={OUT}",
    "--output_name=sola_face_sdxl",
    "--save_model_as=safetensors",
    "--save_precision=fp16",
    "--caption_extension=.txt",
    "--resolution=512,512",
    "--enable_bucket",
    "--min_bucket_reso=256",
    "--max_bucket_reso=1024",
    "--train_batch_size=1",
    "--gradient_checkpointing",
    "--max_train_epochs=10",
    "--save_every_n_epochs=2",
    "--learning_rate=1e-4",
    "--unet_lr=1e-4",
    "--text_encoder_lr=5e-5",
    "--lr_scheduler=cosine",
    "--lr_warmup_steps=50",
    "--optimizer_type=AdamW8bit",
    "--network_module=networks.lora",
    "--network_dim=8",
    "--network_alpha=8",
    "--mixed_precision=fp16",
    "--full_fp16",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--seed=42",
    "--keep_tokens=1",
    "--max_data_loader_n_workers=0",
    "--persistent_data_loader_workers",
]

env = os.environ.copy()
env["PYTHONPATH"] = "/content/sd-scripts" + os.pathsep + env.get("PYTHONPATH", "")
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CMD:", " ".join(cmd))
with open(LOG, "w", encoding="utf-8") as logf:
    logf.write("CMD " + " ".join(cmd) + "\n\n")
    logf.flush()
    proc = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, cwd="/content/sd-scripts")

print("exit_code", proc.returncode)
print("\n===== FULL train.log =====")
print(open(LOG, encoding="utf-8", errors="replace").read())

import glob
paths = glob.glob(OUT + "/**/*.safetensors", recursive=True)
print("\nSAFE:", paths)
if proc.returncode != 0 or not paths:
    raise RuntimeError("Train failed — read FULL train.log printed above and paste it in chat")


In [ ]:
# @title 4) Download
import glob, os
from google.colab import files
paths = sorted(glob.glob("/content/outputs/sola_face_lora/**/*.safetensors", recursive=True), key=os.path.getmtime)
print(paths)
assert paths, "No file"
best = [p for p in paths if "sola_face_sdxl" in os.path.basename(p)] or paths
files.download(best[-1])
